# Week 4 workshop · Cellular automata


# Context


**Canonical model:** Elementary cellular automata and Conway's Game of Life.

**Modelling practice:** Computational complexity grows with the world, runtime, rule search, and number of repeated experiments.

**Why it matters:** A model that is cheap for one small world can become expensive when we enlarge the lattice, run longer, search many rules, or repeat initial conditions. A computational budget is therefore a modelling constraint.

**How it appears here:** We time the same cellular-automaton update at several world sizes and durations, then decide how to allocate a fixed budget between resolution, runtime, rule coverage, and repeated initial conditions. Summary statistics remain one way to compress the resulting output.

By the end, you should be able to estimate a simulation's workload and justify how a fixed computational budget is divided among world size, runtime, rule coverage, and repeated initial conditions.


## From the lecture to the workshop
This workshop contains two related models. Part I uses one-dimensional elementary cellular automata to make rule decoding, full space–time histories and rule-space comparisons explicit. Part II repeats the modelling workflow for Conway's two-dimensional Game of Life, where time must be inspected through snapshots or animation.

The shared computational-budget section then compares the costs of enlarging the world, running longer and repeating experiments. Keeping the two models separate makes it clear which representation and analysis belongs to which state space.


In [ ]:
from typing import Sequence

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

SEED = 3024
INK = "#1B2A4C"
BLUE = "#5879AA"
YELLOW = "#EDCC55"

rng = np.random.default_rng(SEED)
print(f"NumPy {np.__version__} · seed {SEED}")


# Part I · Elementary cellular automata


## Elementary cellular automaton
| Ingredient | Workshop choice |
|---|---|
| World | A one-dimensional lattice |
| State | Each cell is 0 or 1 |
| Neighbourhood | Left, centre, and right cells |
| Boundary | Periodic wrap-around |
| Dynamics | One deterministic rule applied synchronously |
| Initial condition | A single central cell or a reproducible random state |
| Clock | One simultaneous update of every cell |

> **Modelling choice:** State, neighbourhood, boundary, initialisation, and update schedule are separate choices.


Three binary cells form eight possible neighbourhoods. Reading $111,110,\ldots,000$ as binary numbers gives indices $7,6,\ldots,0$.

Rule 90 is

$$90=01011010_2.$$

The bit at index $n$ gives the output for the neighbourhood whose binary value is $n$.


In [ ]:
def rule_table(rule: int) -> np.ndarray:
    """Return outputs indexed by neighbourhood values 0,...,7."""
    if not isinstance(rule, (int, np.integer)) or not 0 <= rule <= 255:
        raise ValueError("rule must be an integer from 0 to 255")
    return np.array([(rule >> index) & 1 for index in range(8)], dtype=np.uint8)


def neighbourhood_bits(value: int) -> tuple[int, int, int]:
    """Return the left, centre, and right bits encoded by a value from 0 to 7."""
    if not isinstance(value, (int, np.integer)) or not 0 <= value <= 7:
        raise ValueError("value must be an integer from 0 to 7")
    return (value >> 2 & 1, value >> 1 & 1, value & 1)


assert np.array_equal(rule_table(90), [0, 1, 0, 1, 1, 0, 1, 0])
assert neighbourhood_bits(6) == (1, 1, 0)

rule = 90
table = rule_table(rule)
for value in range(7, -1, -1):
    print("".join(map(str, neighbourhood_bits(value))), "→", table[value])


> **Discuss:** For Rule 90, which part of the neighbourhood determines the next state? Does the centre matter?

Rule 90 outputs one exactly when the left and right states differ. It is the exclusive-or of the two neighbours.


### Pseudocode · a lookup-table update

```text
FUNCTION elementary_step(current_row, lookup_table):
    CREATE an empty next_row
    FOR each cell i:
        neighbourhood <- (left(i), centre(i), right(i))
        next_row[i] <- lookup_table[neighbourhood]
    RETURN next_row
```

Every entry of `next_row` must be calculated from the same `current_row`.


## Implement and test


### Apply one synchronous update
All outputs must be calculated from the same previous state. Updating cells in place from left to right would define a different, asynchronous model.


In [ ]:
def validate_state(state: np.ndarray) -> np.ndarray:
    """Validate and return a one-dimensional binary state array."""
    state = np.asarray(state)
    if state.ndim != 1 or state.size < 1:
        raise ValueError("state must be a non-empty one-dimensional array")
    if not np.all((state == 0) | (state == 1)):
        raise ValueError("state must contain only zeros and ones")
    return state.astype(np.uint8, copy=False)


def elementary_ca_step(state: np.ndarray, rule: int) -> np.ndarray:
    """Apply one synchronous elementary-CA update with periodic boundaries."""
    state = validate_state(state)
    left = np.roll(state, 1)
    right = np.roll(state, -1)
    neighbourhood_value = 4 * left + 2 * state + right
    return rule_table(rule)[neighbourhood_value]


### Test rules whose behaviour is known
Tests make the bit ordering and periodic boundary convention explicit.


In [ ]:
test_state = np.array([0, 1, 1, 0, 1], dtype=np.uint8)

assert np.array_equal(elementary_ca_step(test_state, 0), np.zeros_like(test_state))
assert np.array_equal(elementary_ca_step(test_state, 255), np.ones_like(test_state))
assert np.array_equal(elementary_ca_step(test_state, 204), test_state)  # identity rule
assert np.array_equal(
    elementary_ca_step(test_state, 90),
    np.bitwise_xor(np.roll(test_state, 1), np.roll(test_state, -1)),
)
assert np.array_equal(test_state, [0, 1, 1, 0, 1])  # input was not mutated

print("All one-step tests passed.")


## Simulate and inspect


**Numerical evolution:** one synchronous step applies the rule to every cell from the same previous configuration.


### Iterate the rule
The returned history includes the initial state at time zero. Rows are time steps and columns are cells.


In [ ]:
def simulate_elementary_ca(
    initial_state: np.ndarray,
    rule: int,
    steps: int,
) -> np.ndarray:
    """Return a state history with shape `(steps + 1, n_cells)`."""
    state = validate_state(initial_state).copy()
    if not isinstance(steps, (int, np.integer)) or steps < 0:
        raise ValueError("steps must be a non-negative integer")
    history = np.empty((steps + 1, state.size), dtype=np.uint8)
    history[0] = state
    for time in range(1, steps + 1):
        state = elementary_ca_step(state, rule)
        history[time] = state
    return history


def single_seed(n_cells: int) -> np.ndarray:
    if not isinstance(n_cells, (int, np.integer)) or n_cells < 1:
        raise ValueError("n_cells must be a positive integer")
    state = np.zeros(n_cells, dtype=np.uint8)
    state[n_cells // 2] = 1
    return state


def random_state(n_cells: int, density: float, rng: np.random.Generator) -> np.ndarray:
    if n_cells < 1 or not 0 <= density <= 1:
        raise ValueError("n_cells must be positive and density must lie in [0,1]")
    return (rng.random(n_cells) < density).astype(np.uint8)


### Show the full space–time history

Animating one row hides the structure that accumulates through time. For a one-dimensional cellular automaton, use the second plotting axis for time so the full trajectory is visible at once.


In [ ]:
initial_single = single_seed(161)
history_110 = simulate_elementary_ca(initial_single, rule=110, steps=120)


In [ ]:
def plot_history(history: np.ndarray, rule: int, ax=None, title: str | None = None):
    """Plot time vertically and cells horizontally."""
    if ax is None:
        _, ax = plt.subplots(figsize=(8, 5))
    ax.imshow(history, cmap="binary", vmin=0, vmax=1, interpolation="nearest", aspect="auto")
    ax.set(xlabel="Cell", ylabel="Time step", title=title or f"Rule {rule}")
    return ax


# Modelling choices: change these values and rerun this cell.
world_size = 201       # number of cells in each row
number_of_steps = 100  # number of updates shown vertically

# Choose an initial condition. Keep one active line and comment out the other.
initial_state = single_seed(world_size)
# initial_state = random_state(
#     world_size,
#     density=0.35,  # expected fraction of cells initially in state 1
#     rng=np.random.default_rng(SEED),
# )

history_90 = simulate_elementary_ca(
    initial_state,
    rule=90,
    steps=number_of_steps,
)
plot_history(history_90, rule=90)
plt.show()


The vertical direction in this image is time, not a second spatial dimension. `world_size` controls the number of cells across each row; `number_of_steps` controls the number of rows of history. An odd world size places the single live seed exactly at the centre.

The initial condition is also a modelling choice. `single_seed(...)` exposes growth from one perturbation. `random_state(..., density=...)` samples a world in which the stated density is the probability that each cell initially equals one. Reusing the same random seed lets us compare rules without also changing the initial world.


## Analyse


### Compare rules fairly
Changing both the rule and the random initial state prevents us from attributing differences to the rule. Generate one initial state, then reuse it.

> **Discuss:** Predict which rule will become uniform, periodic, nested, or persistently irregular.


In [ ]:
shared_initial = random_state(240, density=0.5, rng=np.random.default_rng(SEED))
rules = [0, 4, 30, 90, 110, 204]

fig, axes = plt.subplots(2, 3, figsize=(11, 7), sharex=True, sharey=True)
for ax, rule in zip(axes.flat, rules):
    history = simulate_elementary_ca(shared_initial, rule, steps=160)
    plot_history(history, rule, ax=ax)
fig.tight_layout()
plt.show()


### Related rules and seed-dependent agreement
The 256 rule numbers overcount genuinely different dynamics because some rules differ only by a change of viewpoint:

- **reflection** reverses left and right;
- **state conjugation** swaps 0 and 1 in both the input and output;
- combining both operations gives a third symmetry.

For example, Rules 30 and 86 are reflection partners. Their histories match after reflecting both the initial condition and the displayed history. Reflection and state conjugation reduce the 256 elementary rules to 88 symmetry classes.

This is stronger than two rules agreeing for one seed. A particular trajectory may visit only some of the eight neighbourhoods, so two genuinely different rule tables can produce the same finite history until a neighbourhood on which they disagree is encountered.


In [ ]:
comparison_rng = np.random.default_rng(SEED + 20)
comparison_seed = random_state(121, density=0.45, rng=comparison_rng)

history_30 = simulate_elementary_ca(comparison_seed, rule=30, steps=80)
history_86 = simulate_elementary_ca(comparison_seed[::-1], rule=86, steps=80)

assert np.array_equal(history_30[:, ::-1], history_86)

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
plot_history(history_30, rule=30, ax=axes[0], title="Rule 30")
plot_history(history_86[:, ::-1], rule=86, ax=axes[1], title="Reflected Rule 86 history")
fig.tight_layout()
plt.show()


> **Discuss:** If two rules produce the same history from one initial state, what further test would distinguish accidental agreement from a symmetry of the complete rule?


### Quantitative views
For a binary state $x$:

- **density** $\rho=\langle x\rangle$ is the fraction of cells in state one;
- **binary entropy** $H(\rho)$ measures uncertainty in the state of a randomly sampled cell;
- **activity** is the fraction of cells that change between consecutive steps.

These compress different features. None is a complete measure of spatial complexity.


In [ ]:
def binary_entropy_from_density(density: np.ndarray) -> np.ndarray:
    """Binary Shannon entropy in bits, with 0 log 0 defined as zero."""
    density = np.asarray(density, dtype=float)
    if np.any((density < 0) | (density > 1)):
        raise ValueError("density must lie in [0,1]")
    entropy = np.zeros_like(density)
    interior = (density > 0) & (density < 1)
    p = density[interior]
    entropy[interior] = -(p * np.log2(p) + (1 - p) * np.log2(1 - p))
    return entropy


def ca_observables(history: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Return density, binary state entropy, and between-step activity."""
    history = np.asarray(history)
    density = history.mean(axis=1)
    entropy = binary_entropy_from_density(density)
    activity = np.r_[np.nan, np.mean(history[1:] != history[:-1], axis=1)]
    return density, entropy, activity


density, entropy, activity = ca_observables(history_110)
fig, axes = plt.subplots(3, 1, figsize=(8, 6), sharex=True)
for ax, values, label in zip(axes, (density, entropy, activity), ("Density", "Binary state entropy, H(rho)", "Activity")):
    ax.plot(values, color=INK, linewidth=1.6)
    ax.set_ylabel(label)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", color="#C7CEDC", linewidth=0.6)
axes[-1].set_xlabel("Time step")
fig.tight_layout()
plt.show()


> **Discuss:** Can a static configuration and a rapidly changing configuration have the same density and binary state entropy? Which observable notices the difference?

Rule 204 is a useful counterexample: it can retain high binary state entropy while having zero activity after initialisation. Here $H(\rho)$ is calculated directly from density, so it is not an independent measure of spatial organisation. Entropy and information measures are developed properly in Week 9.


### Map rule space with summary statistics
Apply every rule to a small ensemble of random initial states. For each run, summarise the final 40 steps using three deliberately simple observables:

- **late-time density:** the fraction of cells in state one;
- **late-time activity:** the fraction changing state between steps;
- **neighbour disagreement:** the fraction of adjacent pairs in different states.

The plot moves up the ladder by replacing each complete history with three numbers. It is a map of finite experiments, not a universal classification of the rules.


In [ ]:
def neighbour_disagreement(state: np.ndarray) -> float:
    """Fraction of adjacent pairs that differ, using the periodic boundary."""
    state = np.asarray(state)
    return float(np.mean(state != np.roll(state, 1)))


n_initial_states = 6
late_window = 40
initial_rng = np.random.default_rng(SEED + 1)
initial_states = [random_state(240, density=0.5, rng=initial_rng) for _ in range(n_initial_states)]

summary = []
for rule in range(256):
    run_summaries = []
    for initial_state in initial_states:
        history = simulate_elementary_ca(initial_state, rule, steps=160)
        density, _, activity = ca_observables(history)
        disagreement = np.array([neighbour_disagreement(state) for state in history])
        run_summaries.append((
            np.mean(density[-late_window:]),
            np.nanmean(activity[-late_window:]),
            np.mean(disagreement[-late_window:]),
        ))
    summary.append((rule, *np.mean(run_summaries, axis=0)))

summary = np.asarray(summary)
fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(
    summary[:, 1], summary[:, 2], c=summary[:, 3], cmap="cividis", s=26, alpha=0.82
)
label_offsets = {0: (6, 6), 30: (8, 8), 90: (-20, 9), 110: (8, -13), 204: (-27, 7), 255: (-22, 7)}
for rule, offset in label_offsets.items():
    row = summary[rule]
    ax.annotate(
        str(rule), row[1:3], xytext=offset, textcoords="offset points",
        fontsize=8, arrowprops={"arrowstyle": "-", "color": "#667085", "lw": 0.6},
    )
ax.set(
    xlabel="Late-time mean density",
    ylabel="Late-time mean activity",
    title="A deliberately lossy map of elementary CA rule space",
)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(color="#C7CEDC", linewidth=0.6, alpha=0.7)
fig.colorbar(scatter, ax=ax, label="Late-time neighbour disagreement")
fig.tight_layout()
plt.show()


> **Up the ladder:** one local rule → an ensemble of histories → three observables → a finite survey of rule space.

> **Discuss:** Find two visibly different histories that lie near one another in this map. What information did density, activity, and neighbour disagreement discard?

The survey depends on lattice size, boundary condition, initial-condition ensemble, runtime, averaging window, and chosen observables. A different experiment can move a rule elsewhere in this plot. Neighbour disagreement notices local alternation, but it still cannot distinguish all spatial structures. Week 9 develops entropy and information-based measures that can describe structure more carefully.


# Part II · Conway's Game of Life


## Conway's Game of Life
Game of Life keeps binary states and synchronous updates, but moves to a two-dimensional square lattice. Each cell reads its eight-cell **Moore neighbourhood**.

| Ingredient | Workshop choice |
|---|---|
| World | A two-dimensional square lattice |
| State | Dead (0) or alive (1) |
| Neighbourhood | Eight surrounding cells; the focal cell is not counted |
| Boundary | Periodic wrap-around for the baseline |
| Update | Synchronous B3/S23 rule |

**B3/S23** means that a dead cell is **born** when its neighbour count belongs to $B=\{3\}$, while a live cell **survives** when its count belongs to $S=\{2,3\}$. Every other cell is dead in the next state.

The code below exposes `birth` and `survival` as parameters. Conway's rule is `birth=(3,)`, `survival=(2, 3)`. To test **B4/S2**, use `birth=(4,)`, `survival=(2,)`; the trailing comma makes a one-entry Python tuple.


### Pseudocode · a decision rule

```text
FOR each cell:
    neighbours <- number alive in its Moore neighbourhood
    IF cell is dead AND neighbours = 3:
        next state <- alive
    ELSE IF cell is alive AND neighbours is 2 or 3:
        next state <- alive
    ELSE:
        next state <- dead
REPLACE the grid with the completed next grid
```

The two cellular automata share a synchronous clock, but their worlds, neighbourhoods, and rule representations differ.


## Implement and test


In [ ]:
def validate_life_grid(grid: np.ndarray) -> np.ndarray:
    """Validate and return a two-dimensional binary grid."""
    grid = np.asarray(grid)
    if grid.ndim != 2 or min(grid.shape) < 1:
        raise ValueError("grid must be a non-empty two-dimensional array")
    if not np.all((grid == 0) | (grid == 1)):
        raise ValueError("grid must contain only zeros and ones")
    return grid.astype(np.uint8, copy=False)


def moore_neighbour_count(grid: np.ndarray) -> np.ndarray:
    """Count live Moore neighbours using periodic boundaries."""
    grid = validate_life_grid(grid)
    count = np.zeros_like(grid, dtype=np.uint8)
    for row_shift in (-1, 0, 1):
        for col_shift in (-1, 0, 1):
            if row_shift == 0 and col_shift == 0:
                continue
            count += np.roll(grid, shift=(row_shift, col_shift), axis=(0, 1))
    return count


def life_like_step(
    grid: np.ndarray,
    birth: tuple[int, ...] = (3,),
    survival: tuple[int, ...] = (2, 3),
) -> np.ndarray:
    """Apply one synchronous B/S update with periodic boundaries."""
    grid = validate_life_grid(grid)
    if any(count not in range(9) for count in (*birth, *survival)):
        raise ValueError("birth and survival counts must be integers from 0 to 8")
    neighbours = moore_neighbour_count(grid)
    born = (grid == 0) & np.isin(neighbours, birth)
    survives = (grid == 1) & np.isin(neighbours, survival)
    return (born | survives).astype(np.uint8)


def game_of_life_step(grid: np.ndarray) -> np.ndarray:
    """Apply Conway's B3/S23 rule."""
    return life_like_step(grid, birth=(3,), survival=(2, 3))


def simulate_life_like(
    initial_grid: np.ndarray,
    steps: int,
    birth: tuple[int, ...] = (3,),
    survival: tuple[int, ...] = (2, 3),
) -> np.ndarray:
    """Return a Life-like history including the initial configuration."""
    grid = validate_life_grid(initial_grid).copy()
    if not isinstance(steps, (int, np.integer)) or steps < 0:
        raise ValueError("steps must be a non-negative integer")
    history = np.empty((steps + 1, *grid.shape), dtype=np.uint8)
    history[0] = grid
    for time in range(1, steps + 1):
        grid = life_like_step(grid, birth=birth, survival=survival)
        history[time] = grid
    return history


def simulate_game_of_life(initial_grid: np.ndarray, steps: int) -> np.ndarray:
    """Return a Conway B3/S23 history including the initial configuration."""
    return simulate_life_like(initial_grid, steps, birth=(3,), survival=(2, 3))


### Test the rule using known objects
A visual resemblance is not enough to validate the code. Still lifes,
oscillators and spaceships provide executable checks with known periods and
motions.


In [ ]:
def place_pattern(shape: tuple[int, int], pattern: np.ndarray, top: int, left: int) -> np.ndarray:
    """Place a binary pattern on an otherwise empty periodic grid."""
    grid = np.zeros(shape, dtype=np.uint8)
    pattern = validate_life_grid(pattern)
    rows, cols = pattern.shape
    if top < 0 or left < 0 or top + rows > shape[0] or left + cols > shape[1]:
        raise ValueError("pattern must fit inside the grid")
    grid[top:top + rows, left:left + cols] = pattern
    return grid


BLOCK = np.array([[1, 1], [1, 1]], dtype=np.uint8)
BLINKER = np.array([[1, 1, 1]], dtype=np.uint8)
GLIDER = np.array([[0, 1, 0],
                   [0, 0, 1],
                   [1, 1, 1]], dtype=np.uint8)

block = place_pattern((12, 12), BLOCK, 5, 5)
assert np.array_equal(game_of_life_step(block), block)

blinker = place_pattern((12, 12), BLINKER, 5, 4)
assert not np.array_equal(game_of_life_step(blinker), blinker)
assert np.array_equal(simulate_game_of_life(blinker, 2)[-1], blinker)

glider = place_pattern((20, 20), GLIDER, 5, 5)
after_four = simulate_game_of_life(glider, 4)[-1]
assert np.array_equal(after_four, np.roll(glider, shift=(1, 1), axis=(0, 1)))

print("Block, blinker, and glider tests passed.")


## Simulate and inspect


### Inspect a two-dimensional trajectory

Both plotting axes now represent space, so time cannot be displayed as the second axis as it was for an elementary cellular automaton. Use snapshots or an animation, with direct control over the current frame.

The Game of Life animation is retained and uses Matplotlib's standard controls. A custom web player would not add anything needed for this investigation.


In [ ]:
life_rng = np.random.default_rng(SEED)
life_initial = (life_rng.random((60, 60)) < 0.20).astype(np.uint8)
life_history = simulate_game_of_life(life_initial, steps=120)

snapshot_times = [0, 1, 10, 40, 80, 120]
fig, axes = plt.subplots(2, 3, figsize=(9, 6))
for ax, time in zip(axes.flat, snapshot_times):
    ax.imshow(life_history[time], cmap="binary", vmin=0, vmax=1, interpolation="nearest")
    ax.set_title(f"t = {time}")
    ax.set_xticks([])
    ax.set_yticks([])
fig.tight_layout()
plt.show()


In [ ]:
def simple_life_animation(
    history: np.ndarray,
    *,
    interval: int = 140,
) -> FuncAnimation:
    """Animate a Game-of-Life history with standard Matplotlib controls."""
    history = np.asarray(history, dtype=np.uint8)
    if history.ndim != 3:
        raise ValueError("history must have shape (time, rows, columns)")

    fig, ax = plt.subplots(figsize=(4.2, 3.6))
    image = ax.imshow(
        history[0],
        cmap="binary",
        vmin=0,
        vmax=1,
        interpolation="nearest",
    )
    title = ax.set_title("$t=0$")
    ax.set(xticks=[], yticks=[], xlabel="grid column", ylabel="grid row")

    def update(frame_number: int):
        image.set_data(history[frame_number])
        title.set_text(f"$t={frame_number}$")
        return image, title

    animation = FuncAnimation(
        fig,
        update,
        frames=len(history),
        interval=interval,
        blit=False,
    )
    plt.close(fig)
    return animation


display(HTML("<style>input[value='reflect'], input[value='reflect'] + label {display:none!important}</style>" + simple_life_animation(life_history).to_jshtml(default_mode="once")))


## Analyse


### Summarise one Game of Life run
The animation shows morphology and motion. Two simple time series answer narrower questions:

- **live-cell density** records how much of the grid is occupied;
- **activity** records the fraction of cells that change in one update.

Neither identifies still lifes, oscillators or gliders by itself. Use them alongside the animation rather than as replacements for it.


In [ ]:
def life_observables(history: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Return live-cell density and between-frame activity."""
    history = np.asarray(history, dtype=np.uint8)
    if history.ndim != 3:
        raise ValueError("history must have shape (time, rows, columns)")
    density = history.mean(axis=(1, 2))
    activity = np.r_[np.nan, np.mean(history[1:] != history[:-1], axis=(1, 2))]
    return density, activity


life_density, life_activity = life_observables(life_history)
fig, axes = plt.subplots(2, 1, figsize=(7.5, 4.6), sharex=True)
axes[0].plot(life_density, color=INK, lw=1.8)
axes[0].set_ylabel("Live-cell density")
axes[1].plot(life_activity, color=BLUE, lw=1.8)
axes[1].set(xlabel="Time step", ylabel="Activity")
for ax in axes:
    ax.grid(axis="y", color="#C7CEDC", linewidth=0.6)
    ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
plt.show()


### Investigate the two-dimensional model
1. Replace the random state with a block, blinker, glider, or another documented pattern. Does the observed period or displacement match the test?
2. Replace periodic boundaries with fixed-dead boundaries. Which objects distinguish the two choices, and how long must they run before the difference is visible?
3. Compare several random initial densities using the same grid size and runtime. Record live-cell density and activity through time rather than judging only the final picture.
4. Find a small seed with a long transient. Define the event that marks the end of the transient before comparing seeds.
5. Add one morphology-aware observable, such as connected-component count, cluster-size distribution, perimeter, or a box-counting estimate. State what spatial feature it retains that density discards.

> **Modelling choice:** A larger neighbourhood, an asynchronous clock, or a different boundary does not merely tune Conway's model; it defines a different cellular automaton.


# Computational budget


## Treat computation as a finite budget
For either cellular automaton, a rough work estimate is

$$
\text{sites per world} \times \text{time steps} \times
\text{rules or conditions} \times \text{initial states}.
$$

For an elementary cellular automaton, sites per world is the row length. For Game of Life it is rows multiplied by columns, and each update also inspects a larger neighbourhood. The expression is not an exact runtime formula, but it exposes the experimental trade-off. Doubling every choice quickly becomes expensive.

Time the update itself rather than figure rendering. Record the world size, duration, conditions, repeats and a machine-independent work count. A progress bar shows that a sweep is running; it does not make the design sufficient.


In [ ]:
from time import perf_counter


budget_cases = [(100, 100), (200, 100), (200, 200), (400, 200)]
timings = []
for world_size, number_of_steps in budget_cases:
    rng = np.random.default_rng(SEED)
    initial = random_state(world_size, density=0.5, rng=rng)
    start = perf_counter()
    simulate_elementary_ca(initial, rule=30, steps=number_of_steps)
    elapsed = perf_counter() - start
    cell_updates = world_size * number_of_steps
    timings.append((world_size, number_of_steps, cell_updates, elapsed))

print("world  steps  cell updates  elapsed (s)")
for world_size, number_of_steps, cell_updates, elapsed in timings:
    print(f"{world_size:5d}  {number_of_steps:5d}  {cell_updates:12,d}  {elapsed:10.4f}")


# Extend


## Choose an extension
### A · Change the update schedule
Implement asynchronous updates. Compare with the synchronous baseline using the same rule and initial state.

### B · Change the boundary
Implement fixed-zero or reflecting boundaries. Construct a test that distinguishes them from periodic boundaries.

### C · Allocate a fixed computational budget
Choose a maximum number of cell updates. Compare two allocations, such as a larger world with fewer initial states versus a smaller world with a larger ensemble. State which scientific question each allocation answers better.

### D · Test robustness
Repeat selected rules over an ensemble of initial states and lattice sizes. Which conclusions survive?


In [ ]:
# Your extension goes here. Keep the baseline functions unchanged where possible.


# Exit


## Modelling practice checkpoint
Given a fixed update budget, justify how you divided computation among world size, duration, rules, and repeated initial states.


## Exit ticket
In four sentences:

1. Decode one neighbourhood transition from a rule number.
2. Name one boundary or update-schedule choice.
3. Describe one system-level observable.
4. Explain one reason a visual or numerical classification may not generalise.
